# AccessAI - Prototipo v4.0 (4 clases)

Este notebook es una versión alternativa del prototipo basado en YOLO para el dataset ROD.

Objetivo: mantener una pipeline reproducible con 4 clases agrupadas: `Obstaculo_Dinamico`, `Obstaculo_Fijo`, `Barrera_Arquitectonica` e `Infraestructura_Peatonal`.

> Es un prototipo técnico experimental. Las detecciones no deben interpretarse como una validación final de accesibilidad urbana sin un dataset específico y métricas adecuadas.

In [ ]:
from pathlib import Path
import sys

print('=' * 80)
print('ACCESSAI - ENTORNO Y GPU')
print('=' * 80)

print('Python:', sys.executable)

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
    if torch.cuda.is_available():
        device = 0
        print('GPU:', torch.cuda.get_device_name(device))
        props = torch.cuda.get_device_properties(device)
        print(f'VRAM total: {props.total_memory / 1024**3:.2f} GB')
        torch.set_float32_matmul_precision('high')
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
except ModuleNotFoundError:
    print('Faltan dependencias: activa el entorno del proyecto e instala ultralytics y torch.')
    raise

# ============================================================
# ACCESSAI - 01. ENTORNO Y GPU
# ============================================================

from pathlib import Path
import os
import torch

print("=" * 70)
print("ACCESSAI - ENTORNO")
print("=" * 70)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"CUDA de PyTorch: {torch.version.cuda}")
print(f"HIP: {torch.version.hip}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

    props = torch.cuda.get_device_properties(0)

    print(
        f"Memoria GPU: "
        f"{props.total_memory / 1024**3:.2f} GB"
    )

    print(
        f"Compute capability: "
        f"{props.major}.{props.minor}"
    )

    print(
        f"Memoria reservada actualmente: "
        f"{torch.cuda.memory_reserved(0) / 1024**3:.2f} GB"
    )

    print(
        f"Memoria asignada actualmente: "
        f"{torch.cuda.memory_allocated(0) / 1024**3:.2f} GB"
    )
else:
    print("No hay GPU disponible.")

print("=" * 70)


# [markdown]
# # 02. Comprobar Ultralytics

# ============================================================
# ACCESSAI - 02. ULTRALYTICS
# ============================================================

import ultralytics

print("Ultralytics:", ultralytics.__version__)

!python -m pip show ultralytics

In [ ]:
try:
    import ultralytics
    print('Ultralytics:', ultralytics.__version__)
except ModuleNotFoundError:
    print('Ultralytics no está instalado.')
    raise

In [ ]:
def encontrar_raiz_proyecto():
    actual = Path.cwd().resolve()
    candidatos = [actual] + list(actual.parents)
    for candidato in candidatos:
        dataset = candidato / 'DATA' / 'ROD-Dataset' / 'dataset'
        if dataset.exists():
            return candidato
    return actual

ROOT = encontrar_raiz_proyecto()
DATASET_PATH = ROOT / 'DATA' / 'ROD-Dataset' / 'dataset'
print('ROOT:', ROOT)
print('DATASET_PATH:', DATASET_PATH)
print('Existe dataset:', DATASET_PATH.exists())

yaml_original = DATASET_PATH / 'data.yaml'
yaml_filtrado = DATASET_PATH / 'data_filtrado.yaml'
print('data.yaml:', yaml_original.exists())
print('data_filtrado.yaml:', yaml_filtrado.exists())

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'No existe el dataset: {DATASET_PATH}')

DATA_YAML = yaml_filtrado if yaml_filtrado.exists() else yaml_original
print('YAML usado:', DATA_YAML)

## Mapeo de clases

El dataset original tiene 25 clases; para este prototipo se remapean a 4 categorías experimentales:

- `Obstaculo_Dinamico`
- `Obstaculo_Fijo`
- `Barrera_Arquitectonica`
- `Infraestructura_Peatonal`

In [ ]:
CLASES_ORIGINALES = [
    'Bike',
    'Building',
    'Car',
    'Person',
    'Stairs',
    'Traffic sign',
    'Electrical Pole',
    'Road',
    'Motorcycle',
    'Dustbin',
    'Dog',
    'Manhole',
    'Tree',
    'Guard rail',
    'Pedestrian crosswalk',
    'Truck',
    'Bus',
    'Bench',
    'Traffic Cone',
    'Fire hydrant',
    'Teraffic Barrel',
    'Plant Pot',
    'Electrical Box',
    'Chair',
    'Bicycle Rack',
]

MAPEO_CLASES = {
    0: 0,  # Bike
    2: 0,  # Car
    3: 0,  # Person
    8: 0,  # Motorcycle
    10: 0, # Dog
    15: 0, # Truck
    16: 0, # Bus
    5: 1,  # Traffic sign
    6: 1,  # Electrical Pole
    9: 1,  # Dustbin
    12: 1, # Tree
    13: 1, # Guard rail
    17: 1, # Bench
    18: 1, # Traffic Cone
    19: 1, # Fire hydrant
    20: 1, # Teraffic Barrel
    21: 1, # Plant Pot
    22: 1, # Electrical Box
    23: 1, # Chair
    24: 1, # Bicycle Rack
    4: 2,  # Stairs
    7: 3,  # Road
    11: 3, # Manhole
    14: 3, # Pedestrian crosswalk
}

NUEVOS_NOMBRES = [
    'Obstaculo_Dinamico',
    'Obstaculo_Fijo',
    'Barrera_Arquitectonica',
    'Infraestructura_Peatonal',
]

print('Clases finales:', NUEVOS_NOMBRES)
print('Número de clases:', len(NUEVOS_NOMBRES))

In [ ]:
import yaml
from collections import Counter

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Configuración del dataset:')
print(cfg)
print('Clase objetivo / nc:', cfg.get('nc'))
print('Nombres:', cfg.get('names'))

## Entrenamiento

Este bloque usa `yolo26x.pt` como base y entrena el modelo sobre el dataset filtrado con 4 clases. En entornos con GPU CUDA, conviene usar `batch=-1` para AutoBatch y controlar el uso de VRAM.

In [ ]:
from ultralytics import YOLO
from datetime import datetime

MODEL_NAME = 'yolo26x.pt'
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
RUN_NAME = f'AccessAI_YOLO26X_4clases_{timestamp}'

print('Modelo base:', MODEL_NAME)
print('Train config:', DATA_YAML)
print('Run name:', RUN_NAME)

model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_YAML),
    epochs=20,
    imgsz=640,
    batch=-1,
    project='runs',
    name=RUN_NAME,
    pretrained=True,
    seed=42,
)

In [ ]:
best_model_path = sorted(Path('runs').rglob('best.pt'))[-1]
print('Mejor modelo:', best_model_path)

trained_model = YOLO(str(best_model_path))
metrics = trained_model.val(data=str(DATA_YAML), imgsz=640, split='test')
print(metrics)

## Gráficas y rediseño de análisis

Esta sección genera una vista compacta de las métricas principales del entrenamiento y guarda la gráfica para documentar el experimento.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plot_dir = ROOT / 'resultados'
plot_dir.mkdir(parents=True, exist_ok=True)

metrics_dict = getattr(metrics, 'results_dict', {})
if isinstance(metrics_dict, dict) and metrics_dict:
    keys_to_plot = []
    for key, value in metrics_dict.items():
        lower = key.lower()
        if any(token in lower for token in ['precision', 'recall', 'map', 'f1', 'loss']):
            keys_to_plot.append((key, float(value)))

    if keys_to_plot:
        labels = [k for k, _ in keys_to_plot]
        values = [v for _, v in keys_to_plot]

        plt.figure(figsize=(12, 6))
        bars = plt.bar(labels, values, color=['#4C78A8', '#F58518', '#54A24B', '#E45756', '#B279A2', '#72B7B2'])
        plt.title('Métricas principales de validación - AccessAI v4.0')
        plt.xticks(rotation=30, ha='right')
        plt.ylabel('Valor')
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        for bar, val in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        plt.tight_layout()
        save_plot = plot_dir / 'metricas_accessai_v4.png'
        plt.savefig(save_plot, dpi=180, bbox_inches='tight')
        print(f'Gráfica guardada en: {save_plot}')
        plt.show()
    else:
        print('No se encontraron métricas numéricas para graficar en results_dict.')
else:
    print('El objeto metrics no expone resultados numéricos para crear la gráfica.')

In [ ]:
import json
from pathlib import Path

run_dirs = sorted(Path('runs').glob('AccessAI_YOLO26X_4clases_*'))
if run_dirs:
    latest_run = run_dirs[-1]
    log_file = latest_run / 'results.csv'
    print('Directorio de entrenamiento:', latest_run)
    print('Archivo CSV:', log_file)

    if log_file.exists():
        import pandas as pd
        df = pd.read_csv(log_file)
        if 'epoch' in df.columns and any(col.lower().endswith('loss') for col in df.columns):
            loss_cols = [col for col in df.columns if col.lower().endswith('loss')]
            plt.figure(figsize=(12, 6))
            for col in loss_cols[:3]:
                plt.plot(df['epoch'], df[col], label=col)
            plt.title('Pérdida por epoch - AccessAI v4.0')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            loss_plot = plot_dir / 'loss_por_epoch_accessai_v4.png'
            plt.savefig(loss_plot, dpi=180, bbox_inches='tight')
            print(f'Gráfica de pérdidas guardada en: {loss_plot}')
            plt.show()
        else:
            print('No se encontraron columnas de pérdida en results.csv.')
    else:
        print('No existe results.csv en la última ejecución. Ejecuta primero el entrenamiento.')
else:
    print('No se encontraron ejecuciones en la carpeta runs/.')

## 2. Evolución del mAP por epoch

Esta gráfica compara el progreso de la precisión media por epoch, útil para diagnosticar convergencia del modelo.

In [ ]:
run_dirs = sorted(Path('runs').glob('AccessAI_YOLO26X_4clases_*'))
if run_dirs:
    latest_run = run_dirs[-1]
    log_file = latest_run / 'results.csv'
    if log_file.exists():
        import pandas as pd
        df = pd.read_csv(log_file)
        map_candidates = [c for c in df.columns if 'map' in c.lower() or 'mAP' in c]
        map_candidates = [c for c in map_candidates if '50' in c or '95' in c or 'map' in c.lower()]
        if 'epoch' in df.columns and map_candidates:
            plt.figure(figsize=(12, 6))
            for col in map_candidates[:5]:
                plt.plot(df['epoch'], df[col], label=col)
            plt.title('mAP por epoch - AccessAI v4.0')
            plt.xlabel('Epoch')
            plt.ylabel('mAP')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            map_plot = plot_dir / 'map_por_epoch_accessai_v4.png'
            plt.savefig(map_plot, dpi=180, bbox_inches='tight')
            print(f'Gráfica de mAP guardada en: {map_plot}')
            plt.show()
        else:
            print('No se encontraron columnas de mAP en results.csv.')
    else:
        print('No existe results.csv para graficar el mAP.')
else:
    print('No se encontraron ejecuciones previas para analizar mAP.')

## 3. Resumen final de métricas

Este bloque presenta un resumen compacto con los principales indicadores del modelo y la ruta del mejor checkpoint.

In [ ]:
from pathlib import Path

best_candidates = sorted(Path('runs').rglob('best.pt'))
if best_candidates:
    best_model = best_candidates[-1]
    print('Mejor checkpoint:', best_model)

    if 'metrics' in globals():
        print('\nMétricas observadas:')
        print(metrics)
    else:
        print('El objeto metrics no está definido en este entorno.')
else:
    print('Todavía no existe un checkpoint best.pt en runs/.')

## Inferencia sobre una imagen

Selecciona una imagen en la carpeta de prueba o un archivo nuevo para comprobar la salida del modelo.

In [ ]:
image_path = ROOT / 'tests' / 'imgs' / 'sample.jpg'
output_path = ROOT / 'resultados' / 'accessai_resultado_v4.jpg'
output_path.parent.mkdir(parents=True, exist_ok=True)

if not image_path.exists():
    print('No se encontró la imagen de prueba por defecto:', image_path)
    print('Puedes definir otra ruta como image_path antes de ejecutar la celda.')
else:
    results = trained_model(str(image_path), conf=0.25, save=True)
    print('Resultado de inferencia:', results)
    print('Imagen guardada en:', output_path)

## Nota final

Este notebook representa una versión experimental del prototipo. La detección por clases agrupadas sirve para validar el flujo del pipeline, pero no sustituye una validación real de accesibilidad urbana con un dataset específico y métricas de prueba, como `precision`, `recall`, `F1` y `mAP`.